# ScamSense — Notebook 10: Power BI Dashboard Prep (Kaggle)

Populates Supabase with `model_metrics` and `dataset_overview` tables via the **Supabase REST API** (HTTPS), which works on Kaggle free tier. Direct TCP/port-5432 connections are blocked by Kaggle — this notebook avoids that entirely.

| Cell | What happens |
|---|---|
| 1 | Setup working directory + paths |
| 2 | Install deps + config |
| 3 | Connect to Supabase via REST API |
| 4 | Inspect existing `predictions` table |
| 5 | Create `model_metrics` table (SQL via RPC) |
| 6 | Map columns + populate `model_metrics` |
| 7 | Load + inspect merged dataset |
| 8 | Aggregate + create + populate `dataset_overview` |
| 9 | Verify all 3 tables |
| 10 | Power BI connection instructions |
| 11 | Save outputs |

**Kaggle Secrets required** (Add-ons → Secrets → attach to notebook):
- `SUPABASE_URL` — e.g. `https://vcklncsilpbcwxegtmum.supabase.co`
- `SUPABASE_KEY` — Service Role key from Project Settings → API

> ⚠️ **Why not psycopg2?** Kaggle free notebooks block outbound TCP on port 5432. The Supabase REST API uses HTTPS (port 443) which is allowed.

## Cell 1 — Setup working directory + paths

In [8]:
import os
from pathlib import Path

WORK_DIR  = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')

os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())

print('\nAttached input CSVs:')
for p in sorted(INPUT_DIR.rglob('*.csv'))[:20]:
    print('  ->', p)


Working directory: /kaggle/working

Attached input CSVs:
  -> /kaggle/input/datasets/bhoovika/scamscene/scamsense_full_dataset.csv
  -> /kaggle/input/datasets/bhoovika/scamscene/test.csv
  -> /kaggle/input/datasets/bhoovika/scamscene/train.csv
  -> /kaggle/input/datasets/bhoovika/scamscene/val.csv
  -> /kaggle/input/datasets/bhoovika/scamscene-model/model_comparison.csv


## Cell 2 — Install dependencies + config

In [9]:
import requests
import pandas as pd
import json
from pathlib import Path

# requests + json are pre-installed on Kaggle — no pip needed

INPUT_DIR = Path('/kaggle/input')

def find_input_file(filename):
    matches = list(INPUT_DIR.rglob(filename))
    return matches[0] if matches else None

FULL_DATASET_PATH  = find_input_file('scamsense_full_dataset.csv')
MODEL_METRICS_PATH = find_input_file('model_comparison.csv')

print('Full dataset  :', FULL_DATASET_PATH)
print('Model metrics :', MODEL_METRICS_PATH)

if FULL_DATASET_PATH is None:
    print('\n⚠️  scamsense_full_dataset.csv not found. All CSVs in /kaggle/input:')
    for p in sorted(INPUT_DIR.rglob('*.csv')):
        print('   ->', p)

if MODEL_METRICS_PATH is None:
    print('\n⚠️  eval_metrics.csv not found. Metric-like CSVs:')
    for p in sorted(INPUT_DIR.rglob('*.csv')):
        if any(kw in p.name.lower() for kw in ['metric', 'eval', 'result']):
            print('   ->', p)


Full dataset  : /kaggle/input/datasets/bhoovika/scamscene/scamsense_full_dataset.csv
Model metrics : /kaggle/input/datasets/bhoovika/scamscene-model/model_comparison.csv


## Cell 3 — Connect to Supabase via REST API

**Add two Kaggle secrets** (Add-ons → Secrets → + Add, then toggle Attach to notebook ON):

| Secret name | Where to find it |
|---|---|
| `SUPABASE_URL` | Supabase → Project Settings → API → Project URL |
| `SUPABASE_KEY` | Supabase → Project Settings → API → **service_role** key |

> Use the **service_role** key (not anon) so the notebook can create tables and truncate data.

In [10]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
SUPABASE_URL = secrets.get_secret('SUPABASE_URL').rstrip('/')
SUPABASE_KEY = secrets.get_secret('SUPABASE_KEY')

# Standard headers for all Supabase REST calls
HEADERS = {
    'apikey': SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
    'Content-Type': 'application/json',
    'Prefer': 'return=representation'
}

def supabase_get(table, params=None):
    """SELECT rows from a table via REST."""
    r = requests.get(f'{SUPABASE_URL}/rest/v1/{table}', headers=HEADERS, params=params)
    r.raise_for_status()
    return r.json()

def supabase_post(table, rows):
    """INSERT a list of row dicts into a table."""
    r = requests.post(f'{SUPABASE_URL}/rest/v1/{table}', headers=HEADERS, data=json.dumps(rows))
    r.raise_for_status()
    return r.json()

def supabase_delete(table):
    """DELETE all rows from a table (truncate equivalent)."""
    r = requests.delete(f'{SUPABASE_URL}/rest/v1/{table}?id=gte.0', headers=HEADERS)
    # Also handle non-integer PKs
    if r.status_code == 400:
        r = requests.delete(f'{SUPABASE_URL}/rest/v1/{table}?created_at=gte.1970-01-01', headers=HEADERS)
    return r.status_code

def supabase_rpc(fn_name, params=None):
    """Call a Postgres function via RPC (used for DDL)."""
    r = requests.post(f'{SUPABASE_URL}/rest/v1/rpc/{fn_name}', headers=HEADERS,
                      data=json.dumps(params or {}))
    r.raise_for_status()
    return r

def supabase_sql(sql):
    """Run raw SQL via Supabase pg endpoint (requires service_role key)."""
    r = requests.post(
        f'{SUPABASE_URL}/rest/v1/rpc/exec_sql',
        headers=HEADERS,
        data=json.dumps({'query': sql})
    )
    return r

# Verify connection with a lightweight ping
try:
    resp = requests.get(f'{SUPABASE_URL}/rest/v1/', headers=HEADERS)
    print('Connected to Supabase REST API ✅')
    print('Status:', resp.status_code)
    print('URL:', SUPABASE_URL)
except Exception as e:
    print('❌ Connection failed:', e)


Connected to Supabase REST API ✅
Status: 200
URL: https://vcklncsilpbcwxegtmum.supabase.co


## Cell 4 — Inspect existing `predictions` table

In [11]:
# Fetch first 5 rows to inspect schema
rows = supabase_get('predictions', params={'limit': 5, 'order': 'created_at.desc'})

if rows:
    print('predictions columns:', list(rows[0].keys()))
    df_preview = pd.DataFrame(rows)
    cols = [c for c in ['id', 'created_at', 'language', 'label', 'scam_type', 'risk_tier'] if c in df_preview.columns]
    print(df_preview[cols])
else:
    print('predictions table is empty or does not exist yet')

# Row count via head=true
r = requests.head(f'{SUPABASE_URL}/rest/v1/predictions',
                  headers={**HEADERS, 'Prefer': 'count=exact'})
print(f'\nRow count: {r.headers.get("content-range", "unknown")}')


predictions columns: ['id', 'created_at', 'message', 'language', 'label', 'confidence', 'scam_type', 'risk_score', 'risk_tier', 'explanation', 'sources', 'top_features', 'source_app']
   id                        created_at  language label      scam_type  \
0  12  2026-06-14T11:33:32.881157+00:00        en   ham           None   
1  11  2026-06-14T11:33:32.472525+00:00        ms  scam            job   
2  10  2026-06-14T11:33:31.108735+00:00        ta  scam        unknown   
3   9  2026-06-14T11:33:30.037459+00:00  singlish  scam      ecommerce   
4   8  2026-06-14T11:33:28.629327+00:00        zh  scam  impersonation   

  risk_tier  
0      NONE  
1      HIGH  
2    MEDIUM  
3    MEDIUM  
4      HIGH  

Row count: 0-5/6


## Cell 5 — Create `model_metrics` table

The Supabase REST API cannot run DDL directly. The two options are:
1. **Supabase SQL Editor** (recommended for one-time table creation)
2. A custom `exec_sql` RPC function (created once in the SQL Editor)

**Option 1 — run this SQL once in Supabase SQL Editor** (easiest):
```sql
CREATE TABLE IF NOT EXISTS model_metrics (
    id SERIAL PRIMARY KEY,
    model TEXT,
    language TEXT,
    split TEXT,
    f1 NUMERIC,
    precision_score NUMERIC,
    recall NUMERIC,
    auc NUMERIC,
    created_at TIMESTAMPTZ DEFAULT now()
);
```

**Option 2 — create `exec_sql` RPC once, then use Cell 5b** (enables DDL from Kaggle):
```sql
CREATE OR REPLACE FUNCTION exec_sql(query text)
RETURNS void LANGUAGE plpgsql SECURITY DEFINER AS $$
BEGIN EXECUTE query; END;
$$;
```

In [12]:
# ── Check if model_metrics already exists ────────────────────────────────────
try:
    test = supabase_get('model_metrics', params={'limit': 1})
    print("Table 'model_metrics' exists ✅")
    print('Columns:', list(test[0].keys()) if test else '(empty table)')
except requests.exceptions.HTTPError as e:
    if e.response.status_code == 404:
        print("\u274c Table 'model_metrics' does NOT exist.")
        print('Please create it in the Supabase SQL Editor using the DDL in the markdown above,')
        print('then re-run this cell.')
    else:
        raise

# ── Load + inspect NB05 metrics file ─────────────────────────────────────────
metrics_df = pd.read_csv(MODEL_METRICS_PATH)
print('\nColumns in NB05 metrics file:', list(metrics_df.columns))
print(metrics_df.head(10))


Table 'model_metrics' exists ✅
Columns: ['id', 'model', 'language', 'split', 'f1', 'precision_score', 'recall', 'auc', 'created_at']

Columns in NB05 metrics file: ['model', 'language', 'accuracy', 'f1', 'precision', 'recall', 'auc']
                      model  language  accuracy      f1  precision  recall  \
0      xlmroberta-scamsense   overall    0.9916  0.9916     0.9916  0.9916   
1      xlmroberta-scamsense        en    0.9892  0.9892     0.9892  0.9892   
2      xlmroberta-scamsense        ms    1.0000  1.0000     1.0000  1.0000   
3      xlmroberta-scamsense  singlish    0.9991  0.9991     0.9991  0.9991   
4      xlmroberta-scamsense        ta    0.9993  0.9993     0.9993  0.9993   
5      xlmroberta-scamsense        zh    0.9991  0.9991     0.9991  0.9991   
6  mbert-scamsense-baseline   overall    0.9369  0.9369     0.9369  0.9369   
7  mbert-scamsense-baseline        en    0.9190  0.9190     0.9190  0.9190   
8  mbert-scamsense-baseline        ms    1.0000  1.0000     1.00

## Cell 5b — (Optional) Create tables via exec_sql RPC

Only run this if you set up the `exec_sql` RPC in Supabase. Skip if you created the tables manually.

In [13]:
# Only run if exec_sql RPC exists in your Supabase project

DDL_MODEL_METRICS = """
CREATE TABLE IF NOT EXISTS model_metrics (
    id SERIAL PRIMARY KEY,
    model TEXT,
    language TEXT,
    split TEXT,
    f1 NUMERIC,
    precision_score NUMERIC,
    recall NUMERIC,
    auc NUMERIC,
    created_at TIMESTAMPTZ DEFAULT now()
);
"""

DDL_DATASET_OVERVIEW = """
CREATE TABLE IF NOT EXISTS dataset_overview (
    id SERIAL PRIMARY KEY,
    language TEXT,
    label INTEGER,
    source TEXT,
    scam_type TEXT,
    row_count INTEGER,
    avg_msg_length NUMERIC,
    min_msg_length INTEGER,
    max_msg_length INTEGER,
    created_at TIMESTAMPTZ DEFAULT now()
);
"""

for name, ddl in [('model_metrics', DDL_MODEL_METRICS),
                   ('dataset_overview', DDL_DATASET_OVERVIEW)]:
    r = supabase_sql(ddl)
    if r.status_code in (200, 204):
        print(f"Table '{name}' created/verified ✅")
    else:
        print(f"Table '{name}' error {r.status_code}: {r.text}")
        print('  -> Create manually in Supabase SQL Editor instead.')


Table 'model_metrics' error 404: {"code":"PGRST202","details":"Searched for the function public.exec_sql with parameter query or with a single unnamed json/jsonb parameter, but no matches were found in the schema cache.","hint":null,"message":"Could not find the function public.exec_sql(query) in the schema cache"}
  -> Create manually in Supabase SQL Editor instead.
Table 'dataset_overview' error 404: {"code":"PGRST202","details":"Searched for the function public.exec_sql with parameter query or with a single unnamed json/jsonb parameter, but no matches were found in the schema cache.","hint":null,"message":"Could not find the function public.exec_sql(query) in the schema cache"}
  -> Create manually in Supabase SQL Editor instead.


## Cell 6 — Map columns + populate `model_metrics`

Adjust `COLUMN_MAP` if column names in your metrics CSV differ.

In [14]:
COLUMN_MAP = {
    'model':     'model',
    'language':  'language',
    'split':     'split',
    'f1':        'f1',
    'precision': 'precision_score',   # CSV col -> Supabase col
    'recall':    'recall',
    'auc':       'auc',
}

# Clear existing rows
r = requests.delete(
    f'{SUPABASE_URL}/rest/v1/model_metrics',
    headers={**HEADERS, 'Prefer': 'return=minimal'},
    params={'id': 'gte.0'}
)
print(f'Cleared model_metrics (status {r.status_code})')

# Build row dicts
rows = []
for _, r_row in metrics_df.iterrows():
    row = {}
    for csv_col, db_col in COLUMN_MAP.items():
        val = r_row.get(csv_col) if csv_col in metrics_df.columns else None
        # Convert numpy types to native Python for JSON serialisation
        if val is not None and hasattr(val, 'item'):
            val = val.item()
        row[db_col] = val
    rows.append(row)

# Insert in batches of 500
BATCH = 500
inserted = 0
for i in range(0, len(rows), BATCH):
    supabase_post('model_metrics', rows[i:i+BATCH])
    inserted += len(rows[i:i+BATCH])
    print(f'  Inserted {inserted}/{len(rows)} rows...')

print(f'\nInserted {inserted} rows into model_metrics ✅')


Cleared model_metrics (status 204)
  Inserted 12/12 rows...

Inserted 12 rows into model_metrics ✅


## Cell 7 — Load + inspect merged dataset

In [15]:
full_df = pd.read_csv(FULL_DATASET_PATH, encoding='utf-8-sig')

if 'strat_key' in full_df.columns:
    full_df = full_df.drop(columns=['strat_key'])
if 'scam_type' not in full_df.columns:
    full_df['scam_type'] = None

print('Columns:', list(full_df.columns))
print('Total rows:', len(full_df))
print('\nLanguage counts:\n', full_df['language'].value_counts())
print('\nLabel counts:\n', full_df['label'].value_counts())


Columns: ['text', 'label', 'language', 'source', 'scam_type']
Total rows: 127854

Language counts:
 language
en          97030
ta           9522
zh           7646
singlish     7408
ms           6248
Name: count, dtype: int64

Label counts:
 label
1    63927
0    63927
Name: count, dtype: int64


## Cell 8 — Aggregate + create + populate `dataset_overview`

In [16]:
full_df['msg_length'] = full_df['text'].astype(str).str.len()

GROUP_COLS = ['language', 'label', 'source', 'scam_type']
for col in GROUP_COLS:
    if col not in full_df.columns:
        full_df[col] = None

overview = (
    full_df
    .groupby(GROUP_COLS, dropna=False)
    .agg(row_count=('text', 'count'),
         avg_msg_length=('msg_length', 'mean'),
         min_msg_length=('msg_length', 'min'),
         max_msg_length=('msg_length', 'max'))
    .reset_index()
)
overview['avg_msg_length'] = overview['avg_msg_length'].round(1)
print(f'Aggregated to {len(overview)} rows')
print(overview.head(10))

# Clear existing rows
r = requests.delete(
    f'{SUPABASE_URL}/rest/v1/dataset_overview',
    headers={**HEADERS, 'Prefer': 'return=minimal'},
    params={'id': 'gte.0'}
)
print(f'\nCleared dataset_overview (status {r.status_code})')

# Build row dicts — convert numpy types to native Python
def to_python(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    if hasattr(val, 'item'):
        return val.item()
    return val

rows = [
    {
        'language':       to_python(r['language']),
        'label':          to_python(r['label']),
        'source':         to_python(r['source']),
        'scam_type':      to_python(r['scam_type']),
        'row_count':      to_python(r['row_count']),
        'avg_msg_length': to_python(r['avg_msg_length']),
        'min_msg_length': to_python(r['min_msg_length']),
        'max_msg_length': to_python(r['max_msg_length']),
    }
    for _, r in overview.iterrows()
]

BATCH = 500
inserted = 0
for i in range(0, len(rows), BATCH):
    supabase_post('dataset_overview', rows[i:i+BATCH])
    inserted += len(rows[i:i+BATCH])
    print(f'  Inserted {inserted}/{len(rows)} rows...')

print(f'\nInserted {inserted} rows into dataset_overview ✅')


Aggregated to 24 rows
  language  label                    source  scam_type  row_count  \
0       en      0         job_postings_real        NaN      12344   
1       en      0      phishing_ceas_08_ham        NaN      14187   
2       en      0        phishing_enron_ham        NaN      12907   
3       en      0         phishing_ling_ham        NaN       1971   
4       en      0  phishing_spamassasin_ham        NaN       3366   
5       en      0               uci_sms_ham        NaN       3740   
6       en      1         job_postings_fake        NaN        672   
7       en      1          phishing_ceas_08        NaN      21269   
8       en      1            phishing_enron        NaN      13903   
9       en      1             phishing_ling        NaN        458   

   avg_msg_length  min_msg_length  max_msg_length  
0          1289.8              21           14916  
1          2498.4              11          123029  
2          1625.2               5          180073  
3         

## Cell 9 — Verify all 3 tables

In [17]:
for table in ['predictions', 'model_metrics', 'dataset_overview']:
    try:
        r = requests.head(
            f'{SUPABASE_URL}/rest/v1/{table}',
            headers={**HEADERS, 'Prefer': 'count=exact'}
        )
        count_range = r.headers.get('content-range', '0/?')
        print(f'{table:<20} {count_range} rows')
    except Exception as e:
        print(f'{table:<20} ERROR: {e}')

print('\n' + '='*60)
print('Sample from dataset_overview:')
sample = supabase_get('dataset_overview', params={'limit': 5})
print(pd.DataFrame(sample))

print('\nSample from model_metrics:')
sample = supabase_get('model_metrics', params={'limit': 5})
print(pd.DataFrame(sample))


predictions          0-5/6 rows
model_metrics        0-11/12 rows
dataset_overview     0-23/24 rows

Sample from dataset_overview:
   id language  label                    source scam_type  row_count  \
0  25       en      0         job_postings_real      None      12344   
1  26       en      0      phishing_ceas_08_ham      None      14187   
2  27       en      0        phishing_enron_ham      None      12907   
3  28       en      0         phishing_ling_ham      None       1971   
4  29       en      0  phishing_spamassasin_ham      None       3366   

   avg_msg_length  min_msg_length  max_msg_length  \
0          1289.8              21           14916   
1          2498.4              11          123029   
2          1625.2               5          180073   
3          3151.2              16           28648   
4          1977.4              44          194719   

                         created_at  
0  2026-06-17T05:05:00.842397+00:00  
1  2026-06-17T05:05:00.842397+00:00  
2  

## Cell 10 — Power BI connection instructions

In [18]:
from urllib.parse import urlparse

parsed = urlparse(SUPABASE_URL)
project_ref = parsed.hostname.split('.')[0] if parsed.hostname else '[PROJECT-REF]'

print('='*60)
print('Power BI Desktop -> PostgreSQL connection')
print('(Direct DB connection — use from local machine, not Kaggle)')
print('='*60)
print(f'Server   : db.{project_ref}.supabase.co')
print(f'Port     : 5432')
print(f'Database : postgres')
print(f'Username : postgres')
print('Password : (your Supabase DB password from Project Settings → Database)')
print()
print('Steps:')
print('1. Install the Npgsql driver:')
print('   https://github.com/npgsql/npgsql/releases  -> latest .msi')
print('2. Power BI Desktop -> Get Data -> PostgreSQL database')
print('3. Server / Database as above. Data Connectivity mode: Import')
print('4. Enter Username / Password when prompted')
print('5. Select tables: predictions, model_metrics, dataset_overview -> Load')
print()
print('Alternative: Supabase also offers a Power BI connector via the REST API.')
print(f'REST base URL: {SUPABASE_URL}/rest/v1/')


Power BI Desktop -> PostgreSQL connection
(Direct DB connection — use from local machine, not Kaggle)
Server   : db.vcklncsilpbcwxegtmum.supabase.co
Port     : 5432
Database : postgres
Username : postgres
Password : (your Supabase DB password from Project Settings → Database)

Steps:
1. Install the Npgsql driver:
   https://github.com/npgsql/npgsql/releases  -> latest .msi
2. Power BI Desktop -> Get Data -> PostgreSQL database
3. Server / Database as above. Data Connectivity mode: Import
4. Enter Username / Password when prompted
5. Select tables: predictions, model_metrics, dataset_overview -> Load

Alternative: Supabase also offers a Power BI connector via the REST API.
REST base URL: https://vcklncsilpbcwxegtmum.supabase.co/rest/v1/


## Cell 11 — Save outputs

> **Note:** To push to GitHub, download this notebook from the Output tab and push manually, or use a GitHub Action.

In [19]:
import json
from pathlib import Path

summary = {
    'notebook': 'NB10 Power BI Dashboard Prep',
    'tables_populated': ['predictions', 'model_metrics', 'dataset_overview'],
    'status': 'complete',
    'connection_method': 'Supabase REST API (HTTPS)'
}
out_path = Path('/kaggle/working/nb10_summary.json')
out_path.write_text(json.dumps(summary, indent=2))
print(f'Summary saved -> {out_path}')

print('\n' + '='*60)
print('✅ Notebook 10 complete!')
print('   Tables ready : predictions, model_metrics, dataset_overview')
print('   Next         : Open Power BI Desktop, connect using Cell 10 details')
print('   Output files : /kaggle/working/ (visible in Output tab)')
print('='*60)


Summary saved -> /kaggle/working/nb10_summary.json

✅ Notebook 10 complete!
   Tables ready : predictions, model_metrics, dataset_overview
   Next         : Open Power BI Desktop, connect using Cell 10 details
   Output files : /kaggle/working/ (visible in Output tab)
